# Novo Cabrais (RS): Censo 2010 x Censo 2022

Compara todos os indicadores municipais que o IBGE publica nos dois Censos de forma comparável, para **Novo Cabrais**, o **Rio Grande do Sul** e o **Brasil**.

Temas: população, idade, urbano/rural, cor ou raça, domicílios, saneamento, alfabetização, trabalho e renda, e economia (PIB).

Tudo vem da API do SIDRA (apisidra.ibge.gov.br), que funciona no Colab. Cada número é baixado com uma consulta própria, o que deixa o código simples de conferir. O download completo leva cerca de 1 a 2 minutos.

Valores em reais de 2010 são corrigidos para reais de julho de 2022 pelo IPCA, para comparar poder de compra.

In [ ]:
import re, time, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_rows', 200)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

LOCAIS = {'Novo Cabrais': 'n6/4313391', 'RS': 'n3/43', 'Brasil': 'n1/1'}
API = 'https://apisidra.ibge.gov.br/values'
_cache = {}

def sidra(caminho, tentativas=4):
    """Baixa uma consulta do SIDRA e devolve um DataFrame com as colunas nomeadas pelo cabeçalho."""
    url = f'{API}/{caminho.strip("/")}'
    if url in _cache:
        return _cache[url]
    for i in range(tentativas):
        try:
            r = requests.get(url, timeout=120)
            if r.status_code == 400:
                raise ValueError(f'consulta inválida: {r.text[:200]}')
            r.raise_for_status()
            dados = r.json()
            df = pd.DataFrame(dados[1:]).rename(columns=dados[0])
            df['Valor'] = pd.to_numeric(df['Valor'], errors='coerce')
            _cache[url] = df
            return df
        except ValueError:
            raise
        except Exception as e:
            print(f'  tentativa {i+1} falhou: {e}')
            time.sleep(2 ** (i + 1))
    raise RuntimeError(url)

def num(consulta):
    """Soma dos valores da consulta para cada local. '-' do IBGE (zero) vira 0."""
    return pd.Series({nome: sidra(f'{consulta}/{nivel}')['Valor'].fillna(0).sum() for nome, nivel in LOCAIS.items()})

def valor(consulta):
    """Um valor único (média, mediana, taxa) para cada local; NaN se o IBGE não publicou."""
    return pd.Series({nome: sidra(f'{consulta}/{nivel}')['Valor'].iloc[0] for nome, nivel in LOCAIS.items()})

# Inflação: IPCA de julho de 2010 a julho de 2022 (datas de referência dos dois Censos)
ipca = sidra('t/1737/v/2266/p/201007,202207/n1/1')['Valor'].values
INFLACAO = ipca[1] / ipca[0]
print(f'Preços de jul/2022 = {INFLACAO:.3f} x preços de jul/2010')

## 1. População e idade

In [ ]:
def idades(ano):
    """População por idade simples (0 a 100+) e sexo, para cada local."""
    if ano == 2010:
        base = 't/1378/v/93/p/2010/c1/0/c455/0/c287/all'
        sexos = {'Total': 'c2/0', 'Homens': 'c2/4', 'Mulheres': 'c2/5'}
    else:
        base = 't/9514/v/93/p/2022/c286/113635/c287/all'
        sexos = {'Total': 'c2/6794', 'Homens': 'c2/4', 'Mulheres': 'c2/5'}
    out = []
    for sexo, cs in sexos.items():
        for nome, nivel in LOCAIS.items():
            df = sidra(f'{base}/{cs}/{nivel}')
            col = next(c for c in df.columns if df[c].astype(str).eq('Menos de 1 ano').any())
            for cat, v in zip(df[col], df['Valor'].fillna(0)):
                m = re.fullmatch(r'(\d+) anos?( ou mais)?', cat)
                if cat == 'Menos de 1 ano':
                    out.append((nome, sexo, 0, v))
                elif m and (not m.group(2) or m.group(1) == '100'):   # '70 anos ou mais' é grupo, não idade
                    out.append((nome, sexo, int(m.group(1)), v))
    t = pd.DataFrame(out, columns=['local', 'sexo', 'idade', 'pessoas'])
    return t.groupby(['local', 'sexo', 'idade'])['pessoas'].sum()

idade = {2010: idades(2010), 2022: idades(2022)}

def ind_idade(s):
    s = s.xs('Total', level='sexo')
    r = {}
    for local in LOCAIS:
        p = s.loc[local]
        tot = p.sum()
        jovens, ativos, idosos = p.loc[0:14].sum(), p.loc[15:64].sum(), p.loc[65:].sum()
        acum = p.cumsum() / tot
        r[local] = {
            'População (soma das idades)': tot,
            '% de 0 a 14 anos': 100 * jovens / tot,
            '% de 15 a 64 anos': 100 * ativos / tot,
            '% de 65 anos ou mais': 100 * idosos / tot,
            'Idade mediana (anos)': acum[acum >= .5].index[0],
            'Índice de envelhecimento (idosos 65+ por 100 jovens 0-14)': 100 * idosos / jovens,
            'Razão de dependência (0-14 e 65+ por 100 de 15-64)': 100 * (jovens + idosos) / ativos,
        }
    return pd.DataFrame(r).T

pd.concat({2010: ind_idade(idade[2010]), 2022: ind_idade(idade[2022])}, axis=1).T.round(1)

In [ ]:
# Pirâmide etária de Novo Cabrais, 2010 x 2022 (grupos de 5 anos, % da população)
fig, ax = plt.subplots(figsize=(9, 7))
for ano, cor, estilo in [(2022, '#c0392b', 'bar'), (2010, '#2c3e50', 'contorno')]:
    s = idade[ano].loc['Novo Cabrais']
    tot = s.xs('Total').sum()
    g = lambda sexo: s.xs(sexo).groupby(lambda i: min(i // 5, 18)).sum() / tot * 100
    h, m = g('Homens'), g('Mulheres')
    y = np.arange(len(h))
    if estilo == 'bar':
        ax.barh(y, -h, color=cor, alpha=.6, label=f'{ano}')
        ax.barh(y, m, color=cor, alpha=.6)
    else:
        ax.barh(y, -h, fill=False, edgecolor=cor, lw=1.2, label=f'{ano} (contorno)')
        ax.barh(y, m, fill=False, edgecolor=cor, lw=1.2)
rot = [f'{5*i}-{5*i+4}' for i in range(18)] + ['90+']
ax.set_yticks(range(19)); ax.set_yticklabels(rot)
ax.axvline(0, c='k', lw=.5)
ax.set_xticks(ax.get_xticks()); ax.set_xticklabels([f'{abs(x):.0f}%' for x in ax.get_xticks()])
ax.set_title('Novo Cabrais: pirâmide etária (esquerda homens, direita mulheres)')
ax.legend(loc='upper right'); plt.tight_layout(); plt.show()

## 2. Tabela geral de indicadores

In [ ]:
linhas = []

def add(tema, nome, v2010, v2022, obs=''):
    for local in LOCAIS:
        linhas.append({'tema': tema, 'indicador': nome, 'local': local,
                       '2010': v2010[local], '2022': v2022[local], 'obs': obs})

pct = lambda parte, todo: 100 * parte / todo

# --- População
P10 = num('t/1378/v/93/p/2010/c1/0/c2/0/c287/0/c455/0')
P22 = num('t/9514/v/93/p/2022/c2/6794/c287/100362/c286/113635')
add('População', 'População total', P10, P22)
add('População', '% de mulheres',
    pct(num('t/1378/v/93/p/2010/c1/0/c2/5/c287/0/c455/0'), P10),
    pct(num('t/9514/v/93/p/2022/c2/5/c287/100362/c286/113635'), P22))
add('População', '% morando na zona urbana',
    pct(num('t/1378/v/93/p/2010/c1/1/c2/0/c287/0/c455/0'), P10),
    pct(num('t/9923/v/93/p/2022/c1/1'), num('t/9923/v/93/p/2022/c1/6795')))
ii = {ano: ind_idade(idade[ano]) for ano in idade}
for col in ['% de 0 a 14 anos', '% de 15 a 64 anos', '% de 65 anos ou mais', 'Idade mediana (anos)',
            'Índice de envelhecimento (idosos 65+ por 100 jovens 0-14)',
            'Razão de dependência (0-14 e 65+ por 100 de 15-64)']:
    add('Idade', col, ii[2010][col], ii[2022][col])

# --- Cor ou raça
for nome, c in [('% branca', '2776'), ('% preta', '2777'), ('% parda', '2779'), ('% negra (preta + parda)', '2777,2779')]:
    add('Cor ou raça', nome,
        pct(num(f't/3175/v/93/p/2010/c86/{c}/c1/0/c2/0/c287/0'), P10),
        pct(num(f't/9605/v/93/p/2022/c86/{c}'), num('t/9605/v/93/p/2022/c86/95251')))

# --- Domicílios e saneamento
add('Domicílios', 'Domicílios particulares permanentes', num('t/3451/v/96/p/2010/c1/6795'), num('t/4712/v/381/p/2022'),
    '2022: somente os ocupados')
add('Domicílios', 'Média de moradores por domicílio', valor('t/3451/v/246/p/2010/c1/6795'), valor('t/4712/v/5930/p/2022'))
D10 = 't/3218/v/96/p/2010'
tot10 = num(f'{D10}/c61/0/c299/0/c67/0/c309/0')
add('Saneamento', '% de domicílios com água da rede geral',
    pct(num(f'{D10}/c61/92853/c299/0/c67/0/c309/0'), tot10),
    pct(num('t/6803/v/381/p/2022/c1821/72144'), num('t/6803/v/381/p/2022/c1821/72129')),
    '2022: ligados à rede e usando como forma principal')
add('Saneamento', '% com esgoto em rede geral ou fossa séptica',
    pct(num(f'{D10}/c61/0/c299/9678,9679/c67/0/c309/0'), tot10),
    pct(num('t/6805/v/381/p/2022/c11558/72110,72111,72112'), num('t/6805/v/381/p/2022/c11558/46292')),
    'Inclui fossa séptica (e fossa filtro em 2022)')
add('Saneamento', '% com lixo coletado',
    pct(num(f'{D10}/c61/0/c299/0/c67/2520/c309/0'), tot10),
    pct(num('t/6892/v/381/p/2022/c67/2520'), num('t/6892/v/381/p/2022/c67/10972')))

# --- Alfabetização 15+
pop15_10 = idade[2010].xs('Total', level='sexo').loc[(slice(None), slice(15, None))].groupby(level='local').sum()
alf15_10 = num('t/3213/v/2591/p/2010/c287/100362') - num('t/3213/v/2591/p/2010/c287/93084,93085')
A22 = 't/9542/v/950/p/2022/c2/6794/c86/95251/c287/100362'
add('Educação', 'Taxa de alfabetização, 15 anos ou mais (%)',
    pct(alf15_10, pop15_10), pct(num(f'{A22}/c59/1023'), num(f'{A22}/c59/93024')))
add('Educação', 'Pessoas de 15+ que não sabem ler e escrever',
    pop15_10 - alf15_10, num(f'{A22}/c59/1024'))

In [ ]:
# --- Trabalho e renda (rendimento de todos os trabalhos, pessoas de 10+ ocupadas)
R10 = 't/3577/p/2010/c1/6795/c1568/120704/c521/14583/c58/95253'
R22 = 't/10291/p/2022/c58/95253'
def renda(ano, var, sexo='total', cor='95251'):
    if ano == 2010:
        s = {'total': '6794', 'h': '4', 'm': '5'}[sexo]
        v = {'ocup': '916', 'media': '1985', 'mediana': '1984'}[var]
        return (num if var == 'ocup' else valor)(f'{R10}/v/{v}/c2/{s}/c86/{cor}')
    s = {'total': '6794', 'h': '4', 'm': '5'}[sexo]
    v = {'ocup': '844', 'media': '846', 'mediana': '845'}[var]
    return (num if var == 'ocup' else valor)(f'{R22}/v/{v}/c2/{s}/c86/{cor}')

add('Trabalho e renda', 'Pessoas ocupadas (10+)', renda(2010, 'ocup'), renda(2022, 'ocup'),
    'NÃO comparável: 2010 conta todos os ocupados (inclusive sem remuneração), 2022 só os com rendimento')
add('Trabalho e renda', 'Renda média do trabalho (R$ de jul/2022)', renda(2010, 'media') * INFLACAO, renda(2022, 'media'),
    '2010 corrigido pelo IPCA')
add('Trabalho e renda', 'Renda mediana do trabalho (R$ de jul/2022)', renda(2010, 'mediana') * INFLACAO, renda(2022, 'mediana'),
    '2010 corrigido pelo IPCA')
add('Trabalho e renda', 'Renda média nominal (R$ da época)', renda(2010, 'media'), renda(2022, 'media'))
add('Trabalho e renda', 'Mulheres ganham, para cada R$ 100 dos homens',
    100 * renda(2010, 'media', 'm') / renda(2010, 'media', 'h'),
    100 * renda(2022, 'media', 'm') / renda(2022, 'media', 'h'))

def razao_racial(ano):
    pr, pa, br = (renda(ano, 'ocup', cor=c) for c in ('2777', '2779', '2776'))
    mp, mpa, mb = (renda(ano, 'media', cor=c) for c in ('2777', '2779', '2776'))
    return 100 * ((mp * pr + mpa * pa) / (pr + pa)) / mb
add('Trabalho e renda', 'Negros ganham, para cada R$ 100 dos brancos', razao_racial(2010), razao_racial(2022))

# --- Economia (PIB dos Municípios)
def pib(ano):
    return num(f't/5938/v/37/p/{ano}')
add('Economia', 'PIB (milhões de R$ de jul/2022)', pib(2010) / 1000 * INFLACAO, pib(2022) / 1000,
    '2010 corrigido pelo IPCA de julho; PIB 2022 sujeito a revisão')
add('Economia', 'PIB por habitante (R$ de jul/2022)', pib(2010) * 1000 / P10 * INFLACAO, pib(2022) * 1000 / P22,
    'População do Censo de cada ano')
add('Economia', '% da economia vinda da agropecuária', valor('t/5938/v/516/p/2010'), valor('t/5938/v/516/p/2021'),
    'Último ano publicado é 2021')

tab = pd.DataFrame(linhas)
tab['variação'] = tab['2022'] - tab['2010']
tab['variação %'] = 100 * tab['variação'] / tab['2010'].abs()
print('Pronto:', tab['indicador'].nunique(), 'indicadores')

In [ ]:
# Visão de Novo Cabrais, com RS e Brasil ao lado
ordem = pd.MultiIndex.from_frame(tab[['tema', 'indicador']].drop_duplicates())
largo = tab.set_index(['tema', 'indicador', 'local'])[['2010', '2022']].unstack('local').swaplevel(axis=1)
largo = largo[[(l, a) for l in LOCAIS for a in ['2010', '2022']]].reindex(ordem)
largo.round(1)

In [ ]:
# Só Novo Cabrais: 2010, 2022 e quanto mudou
nc = tab[tab.local == 'Novo Cabrais'].set_index(['tema', 'indicador'])[['2010', '2022', 'variação', 'variação %', 'obs']]
nc.round(1)

In [ ]:
# Novo Cabrais mudou mais ou menos que o RS e o Brasil? (indicadores em %)
sel = tab[tab['indicador'].str.startswith('%') | tab['indicador'].str.contains('para cada')]
p = sel.pivot_table(index='indicador', columns='local', values='variação', sort=False)[list(LOCAIS)]
ax = p.plot.barh(figsize=(9, 0.45 * len(p) + 1.5), color=['#c0392b', '#7f8c8d', '#bdc3c7'], width=.8)
ax.axvline(0, c='k', lw=.6); ax.invert_yaxis()
ax.set_xlabel('Mudança de 2010 para 2022 (pontos)')
ax.set_title('Quanto cada indicador mudou entre os Censos')
plt.tight_layout(); plt.show()

In [ ]:
# Salvar: planilha com todos os indicadores (aparece na pasta à esquerda do Colab, pronta para baixar)
with pd.ExcelWriter('novo_cabrais_2010_2022.xlsx') as xls:
    nc.round(2).to_excel(xls, sheet_name='Novo Cabrais')
    largo.round(2).to_excel(xls, sheet_name='Comparação RS e Brasil')
    tab.round(2).to_excel(xls, sheet_name='Dados longos', index=False)
print('Arquivo salvo: novo_cabrais_2010_2022.xlsx')

## 3. Outras fontes para mostrar ao prefeito

O que não está nos Censos, mas existe por município e dá para comparar no tempo:

| Tema | Fonte | O que traz |
|---|---|---|
| Agricultura | **Censo Agropecuário 2006 x 2017** (IBGE/SIDRA) | número de estabelecimentos, área, agricultura familiar, idade e escolaridade do produtor, uso de assistência técnica e crédito, tratores |
| Produção agrícola | **PAM e PPM** (IBGE/SIDRA, todo ano) | área plantada e valor da produção de cada lavoura (fumo, soja, arroz, milho...), rebanhos, leite |
| Escolas | **IDEB e Censo Escolar** (INEP) | nota das escolas municipais, matrículas, distorção idade-série, abandono |
| Saúde | **DataSUS** (SINASC, SIM) | nascimentos, mortalidade infantil, causas de morte, cobertura de vacinas |
| Assistência social | **CadÚnico e Bolsa Família** (MDS, VIS Data) | famílias de baixa renda cadastradas, beneficiários |
| Emprego formal | **RAIS e Novo CAGED** (Ministério do Trabalho) | empregos com carteira e salários por setor, idade, sexo e cor |
| Finanças da prefeitura | **Siconfi / FINBRA** (Tesouro Nacional) | receitas, despesas por área (saúde, educação, agricultura) |
| Desenvolvimento | **IDHM** (Atlas Brasil, PNUD) | IDHM de 1991, 2000 e 2010 (o de 2022 ainda não saiu por município) |

O Censo Agropecuário e a PAM também estão na API do SIDRA, então dá para acrescentar ao notebook com a mesma função `num()`.